In [ ]:
# Load system libraries
import os
import gc  # Add garbage collection
import sys
from pathlib import Path

# Load progress bar
from tqdm import tqdm

# Load data manipulation libraries
import numpy as np
import pandas as pd
import itertools

# Load spatial data libraries
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.enums import Resampling
from shapely.geometry import Point, Polygon, box, mapping
from rasterio.windows import from_bounds
from rasterio.features import geometry_mask
from affine import Affine

# Load statistical libraries
from scipy import stats
from scipy.stats import mode
#import statsmodels.api as sm
from scipy.stats import chi2_contingency

# Load data visualisation libraries
import matplotlib.pyplot as plt


from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report, RocCurveDisplay
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.metrics import average_precision_score
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from xgboost import plot_importance


##### *Fires data*

In [2]:
# Read fire data
df_fire = gpd.read_file('data/fires/fires_1999_2022.shp')

# Standardise data types
df_fire['year'] = df_fire['Year'].astype(int)
df_fire['month'] = df_fire['Month'].astype(int)

# Select only necessary columns early to reduce memory
df_fire = df_fire[['geometry', 'year', 'month', 'area_ha']]

##### *Temporal data*

In [3]:
# read teleconnections data
df_oni = pd.read_csv('data_cleaning/data/cleaned/oni/cleaned_data.csv')
df_sst = pd.read_csv('data_cleaning/data/cleaned/sst/cleaned_data.csv')

# add lags to teleconnections
df_oni = pd.read_csv('data_cleaning/data/cleaned/oni/cleaned_data.csv')
df_sst = pd.read_csv('data_cleaning/data/cleaned/sst/cleaned_data.csv')
df_tele = df_oni.merge(df_sst, how='left', on=['year', 'month'])

df_tele = df_tele.sort_values(["year", "month"])

lags = [1, 
        3, 
        6
        ]
tele_cols = ['oniTOTAL', 'oniANOM', 'nino1_plus_2', 'nino_anom', 
             'nino3', 'nino_anom_1', 'nino4', 'nino_anom_2', 
             'nino3_4', 'nino_anom_3']

for col in tele_cols:
    for lag in lags:
        df_tele[f"{col}_lag{lag}"] = df_tele[col].shift(lag)

lagged_tele = df_tele[['year', 'month'] + [col for col in df_tele.columns if 'lag' in col]]

# filter to date range of interest
lagged_tele = lagged_tele[(lagged_tele['year'] >= 1999) & (lagged_tele['year'] <= 2022)]

In [4]:
# Tourism data
df_tourism = pd.read_csv('data_cleaning/data/cleaned/tourism/cleaned_data.csv')
df_tourism = df_tourism.rename(columns={"Year": "year", "Month": "month"})
df_tourism = df_tourism[(df_tourism['Island'] == "STATEWIDE") & (df_tourism['Arrival_Type'] == "Total")]
df_tourism = df_tourism.drop(['Island', 'Arrival_Type'], axis=1)

# Combine temporal data
df_temporal = lagged_tele.merge(df_tourism, how='left', on=['year', 'month'])

# Clean up intermediate dataframes
del df_oni, df_sst, df_tourism
gc.collect()

20

##### *Slope and elevation data*

In [5]:
# read in slope and elevation arrays
elevation_array = np.load('data_cleaning/data/cleaned/elevation/cleaned_data.npy')
slope_array = np.load('data_cleaning/data/cleaned/slope/cleaned_data.npy')

# read in raw slope file to get (shared) metadata - make it the reference metadata
slope_raster = rasterio.open('data/elevation/LF2020_SlpD_220_HI/Tif/LH20_SlpD_220.tif')
ref_meta = slope_raster.meta

ref_transform = ref_meta['transform']
ref_width = ref_meta['width']
ref_height = ref_meta['height']
ref_crs = ref_meta['crs']

##### *Vegetation data*

In [6]:
# Define function to align raster layers
## Realise this is unnecessary, as you can just run it twice with target raster as file 1
def align_rasters_in_memory(src_path1, src_path2, target_meta=None):
    with rasterio.open(src_path1) as src1, rasterio.open(src_path2) as src2:
        # Read data and metadata from the first raster
        data1 = src1.read(1)
        meta1 = src1.meta.copy()

        # Read data from the second raster
        data2 = src2.read(1)
        meta2 = src2.meta.copy()

        # If no target CRS is specified, use raster1's CRS
        if target_meta is None:
            target_meta = meta1

        # Check if reprojecting is necessary for both rasters
        if meta1['crs'] != target_meta['crs'] or meta1['transform'] != target_meta['transform']:
            print(f"Reprojecting raster1 to target CRS: {target_meta['crs']}")

            # Create a destination array with the shape of raster1
            dest_array1 = np.empty((target_meta['height'], target_meta['width']), dtype=np.float32)

            # Reproject raster1 to match target CRS and transform
            reproject(
                source=data1,
                destination=dest_array1,
                src_transform=src1.transform,
                src_crs=src1.crs,
                dst_transform=target_meta['transform'],
                dst_crs=target_meta['crs'],
                resampling=Resampling.nearest
            )

            # Update the metadata for the reprojected raster
            meta1.update({
                'height': target_meta['height'],
                'width': target_meta['width'],
                'transform': target_meta['transform'],
                'crs': target_meta['crs']
            }) 

            aligned_data1 = dest_array1
        else:
            # No reprojection needed
            aligned_data1 = data1
            meta1 = target_meta.copy()


        if meta2['crs'] != target_meta['crs'] or meta2['transform'] != target_meta['transform']:
            print(f"Reprojecting raster2 to target CRS: {target_meta['crs']}")

            # Create a destination array with the shape of raster1
            dest_array2 = np.empty((target_meta['height'], target_meta['width']), dtype=np.float32)

            # Reproject raster1 to match target CRS and transform
            reproject(
                source=data2,
                destination=dest_array2,
                src_transform=src2.transform,
                src_crs=src2.crs,
                dst_transform=target_meta['transform'],
                dst_crs=target_meta['crs'],
                resampling=Resampling.nearest
            )

            # Update the metadata for the reprojected raster
            meta2.update({
                'height': target_meta['height'],
                'width': target_meta['width'],
                'transform': target_meta['transform'],
                'crs': target_meta['crs']
            }) 

            aligned_data2 = dest_array2 
        
        else:
            # No reprojection needed
            aligned_data2 = data2
            meta2 = target_meta.copy()

    return aligned_data1, meta1, aligned_data2, meta2

# Function to downsample using mode, ignoring extra rows and columns
def downsample_mode(data, factor):
    # Determine new dimensions
    new_height = data.shape[0] // factor * factor
    new_width = data.shape[1] // factor * factor

    # Crop the data to be evenly divisible by the factor
    cropped_data = data[:new_height, :new_width]

    # Reshape and compute mode
    reshaped = cropped_data.reshape((new_height // factor, factor, new_width // factor, factor))
    reshaped = reshaped.transpose(0, 2, 1, 3).reshape(-1, factor * factor)
    mode_vals, _ = mode(reshaped, axis=1)
    return mode_vals.reshape(new_height // factor, new_width // factor)

# Function to convert downsampled raster to GeoDataFrame with polygons
def raster_to_gdf_polygons(raster, transform, crs, year):
    # Get dimensions of the downsampled raster
    rows, cols = raster.shape

    # Create an empty list to store polygons and values
    polygons = []
    values = []

    # Calculate the scaling factor in terms of original raster coordinates
    scale_x = transform.a * scaling_factor
    scale_y = transform.e * scaling_factor

    # Iterate through each cell in the downsampled raster
    for row in range(rows):
        for col in range(cols):
            # Calculate the bounding box of the cell in original coordinates
            xmin, ymin = transform * (col * scaling_factor, row * scaling_factor)
            xmax, ymax = transform * ((col + 1) * scaling_factor, (row + 1) * scaling_factor)
            
            # Create the polygon for this cell
            cell_polygon = box(xmin, ymin, xmax, ymax)
            polygons.append(cell_polygon)
            values.append(raster[row, col])

    # Create GeoDataFrame
    gdf = gpd.GeoDataFrame({'veg_year': year, 'value': values}, geometry=polygons, crs=crs)
    return gdf

In [8]:
# Set scaling factor
scaling_factor = 20

# Align raster data
veg_dir_2002 = 'data/vegetation/LF2002_EVT_105_HI/Tif/hi_105evt.tif'
veg_dir_2014 = 'data/vegetation/LF2014_EVT_140_HI/Tif/hi_140evt.tif'
veg_2002, meta1, veg_2014, meta2 = align_rasters_in_memory(veg_dir_2002, veg_dir_2014, target_meta=ref_meta)

# Identify legends
veg_2002_legend = 'data/vegetation/LF2002_EVT_105_HI/CSV_Data/hi_105evt.csv'
veg_2014_legend = 'data/vegetation/LF2014_EVT_140_HI/CSV_Data/hi_140evt.csv'

# Process files and create combined GeoDataFrame
gdf_veg_all = []
for vegetation, year, legend in tqdm(zip([veg_2002, veg_2014], [2002, 2014], [veg_2002_legend, veg_2014_legend]), desc='Processing vegetation data', total=2):
    # Downsample the reprojected data by a factor of 40
    downsampled_vegetation = downsample_mode(vegetation, scaling_factor)
    
    # Convert downsampled raster to GeoDataFrame with polygons
    gdf_veg = raster_to_gdf_polygons(downsampled_vegetation, ref_transform, ref_crs, year)

    # Filter out areas with no data
    gdf_veg = gdf_veg[gdf_veg['value'] != -9999]
    
    # Load legend data
    veg_legend = pd.read_csv(legend)
    veg_legend.columns = [col.lower() for col in list(veg_legend.columns)]
    value_col = veg_legend.columns.get_loc('value')
    veg_legend = veg_legend.iloc[:, value_col:value_col + 2]
    veg_legend.columns = ['value', 'veg_type']
    veg_legend = veg_legend.set_index('value').to_dict()['veg_type']
    
    # Create column for vegetation type
    gdf_veg['veg_type'] = gdf_veg['value'].replace(veg_legend)
    
    # Append the GeoDataFrame to the list
    gdf_veg_all.append(gdf_veg)

# Combine all vegetation GeoDataFrames into one
gdf_veg_all = pd.concat(gdf_veg_all, ignore_index=True)

# Categorise vegetation types
land_use_dict = {
    'water' : ['Open Water', 'Water'],
    'urban_dev': ['Developed-Open Space', 'Developed-Low Intensity', 'Developed-Medium Intensity', 'Developed-High Intensity'],
    'agriculture': ['Agriculture-Cultivated Crops and Irrigated Agriculture', 'Agriculture'],
    'wetland': ['Hawaiian Introduced Wetland Vegetation-Herbaceous', "Hawai'i Introduced Wetland Vegetation-Herbaceous", "Hawai'i Bog"],
    'forested': ["Hawai'i Lowland Mesic Forest", "Hawai'i Lowland Rainforest", "Hawai'i Montane Rainforest",
                 "Hawai'i Montane-Subalpine Mesic Forest", "Hawai'i Montane Cloud Forest", "Hawai'i Lowland Dry Forest",
                 "Hawai'i Montane-Subalpine Dry Forest and Woodland"],
    'shrub_grass': ['Hawaiian Introduced Perennial Grassland', 'Hawaiian Introduced Deciduous Shrubland',
                    "Hawai'i Wet Cliff and Ridge Crest Shrubland", 'Hawaiian Introduced Evergreen Shrubland',
                    "Hawai'i Lowland Dry Shrubland", "Hawai'i Lowland Mesic Shrubland", "Hawai'i Montane-Subalpine Dry Shrubland",
                    "Hawai'i Lowland Mesic Grassland", "Hawai'i Introduced Perennial Grassland",
                    "Hawai'i Introduced Deciduous Shrubland", "Hawai'i Introduced Evergreen Shrubland",
                    "Hawai'i Montane-Subalpine Dry Grassland", "Hawai'i Lowland Dry Grassland", "Hawai'i Subalpine Mesic Shrubland"],
    'barren': ['Barren', "Hawai'i Dry Cliff"],
    'introduced_dry': ['Hawaiian Introduced Dry Forest', "Hawai'i Introduced Dry Forest"],
    'introduced_wet_mesic': ['Hawaiian Introduced Wet-Mesic Forest', "Hawai'i Introduced Wet-Mesic Forest"],
    'plantation': ['Hawaiian Managed Tree Plantation', "Hawai'i Managed Tree Plantation"]}
    # 'eucalpytus': ['Hawaiian Introduced Wet-Mesic Forest',  'Hawaiian Introduced Dry Forest',
    #                'Hawaiian Managed Tree Plantation', "Hawai'i Introduced Wet-Mesic Forest",
    #                "Hawai'i Introduced Dry Forest", "Hawai'i Managed Tree Plantation"]} # wet-mesic, dry forest, tree plantation

# Initialize columns with False values
for land_type in land_use_dict.keys():
    gdf_veg_all[land_type] = 0

# Label data based on land use
for land_type, values in land_use_dict.items():
    gdf_veg_all.loc[gdf_veg_all['veg_type'].isin(values), land_type] = 1

# Drop values with water
gdf_veg_all = gdf_veg_all[gdf_veg_all['water'] == False]
gdf_veg_all = gdf_veg_all.drop('water', axis=1)

# Create empty dataframe with every year and month
spatial_points = list(gdf_veg_all['geometry'].unique())
years = list(range(1999,2023))
months = list(range(1,13))

# Generate all combinations
combinations = list(itertools.product(spatial_points, years, months))
df = pd.DataFrame(combinations, columns=['geometry', 'year', 'month'])

# Convert 'geometry' column to shapely Points
df['geometry'] = df['geometry'].apply(lambda x: Polygon(x))

# Create GeoDataFrame
gdf = gpd.GeoDataFrame(df, geometry='geometry', crs=gdf_veg_all.crs)

# Add 2002 vegetation data for all years less than or equal to 2008
gdf_2002 = gdf[gdf['year'] <= 2008].merge(gdf_veg_all[gdf_veg_all['veg_year'] == 2002], on='geometry', how='left')

# Add 2014 vegetation data for all years greater than 2008
gdf_2014 = gdf[gdf['year'] > 2008].merge(gdf_veg_all[gdf_veg_all['veg_year'] == 2014], on='geometry', how='left')

# Concatentate data
gdf = pd.concat([gdf_2002, gdf_2014], axis=0)

Reprojecting raster1 to target CRS: ESRI:102007
Reprojecting raster2 to target CRS: ESRI:102007


Processing vegetation data: 100%|██████████| 2/2 [00:33<00:00, 16.96s/it]


#### Combine data

In [9]:
# add summary statistics of slope and elevation to each veg polygon
# function to crop raster to a bounding box surrounding the fire polygon
def crop_raster_window(raster, transform, polygon):
    # Get polygon bounds
    minx, miny, maxx, maxy = polygon.bounds

    # Convert bounds to pixel coordinates
    window = from_bounds(minx, miny, maxx, maxy, transform=transform)
    
    # Convert float window offsets to ints for slicing
    row_start = max(0, int(window.row_off))
    row_stop = min(raster.shape[0], int(window.row_off + window.height))
    col_start = max(0, int(window.col_off))
    col_stop = min(raster.shape[1], int(window.col_off + window.width))

    # Extract window from raster array
    cropped = raster[row_start:row_stop, col_start:col_stop]
    
    # Compute the new transform for cropped window
    new_transform = transform * Affine.translation(col_start, row_start)

    return cropped, new_transform

# function to calculate summary stats within fire polygons
def get_polygon_stats(raster, polygon, transform):
    # Create mask: True inside polygon
    mask = geometry_mask(
        [mapping(polygon)],
        invert=True,
        out_shape=raster.shape,
        transform=transform
    )
    values = raster[mask]

    # Handle case of empty mask or no data values
    values = values[~np.isnan(values)]
    if values.size == 0:
        return {
            'mean': np.nan,
            'median': np.nan,
            'min': np.nan,
            'max': np.nan,
            'std': np.nan
        }

    return {
        'mean': np.mean(values),
        'median': np.median(values),
        'min': np.min(values),
        'max': np.max(values),
        'std': np.std(values)
    }

elev_stats = []
slope_stats = []

for polygon in spatial_points:
    cropped_elev, cropped_elev_transform = crop_raster_window(elevation_array, ref_transform, polygon)
    cropped_slope, cropped_slope_transform = crop_raster_window(slope_array, ref_transform, polygon)
    elev_stats.append(get_polygon_stats(cropped_elev, polygon, cropped_elev_transform))
    slope_stats.append(get_polygon_stats(cropped_slope, polygon, cropped_slope_transform))

elev_df = pd.DataFrame(elev_stats).add_prefix('elev_').reset_index(drop=True)
slope_df = pd.DataFrame(slope_stats).add_prefix('slope_').reset_index(drop=True)

# combine with geometries
polygons_slope_elev = pd.DataFrame(spatial_points, columns=["geometry"])
polygons_slope_elev = pd.concat([polygons_slope_elev, elev_df, slope_df], axis=1)

# merge slope and elevation onto veg gdf
gdf = gdf.merge(polygons_slope_elev, how='left', on='geometry')

In [10]:
# Align CRS of fires
df_fire = df_fire.to_crs(ref_crs)

# Left join fires to raster points
gdf_final = gpd.sjoin(gdf, df_fire, how='left', predicate='intersects', rsuffix='fire', lsuffix='ref').reset_index()

# Set area to zero if year and month do not match
gdf_final['area_ha'] = np.where(
    (gdf_final['year_ref'] != gdf_final['year_fire']) | (gdf_final['month_ref'] != gdf_final['month_fire']),
    np.nan, gdf_final['area_ha'])

# Remove duplicated matches
gdf_final = gdf_final[~gdf_final.duplicated(subset=['year_ref', 'month_ref', 'geometry', 'area_ha'], keep='first')]
extra_rows = gdf_final[gdf_final.duplicated(subset=['year_ref', 'month_ref', 'geometry'], keep=False)]
extra_rows_no_fire = extra_rows[extra_rows['area_ha'].isna()]
gdf_final = gdf_final.drop(extra_rows_no_fire.index)

In [ ]:
# Align CRS of fires
df_fire = df_fire.to_crs(ref_crs)

# Left join fires to raster points
gdf_final = gpd.sjoin(gdf, df_fire, how='left', predicate='intersects', rsuffix='fire', lsuffix='ref').reset_index()

# Set area to zero if year and month do not match
gdf_final['area_ha'] = np.where(
    (gdf_final['year_ref'] != gdf_final['year_fire']) | (gdf_final['month_ref'] != gdf_final['month_fire']),
    np.nan, gdf_final['area_ha'])

# Remove duplicated matches
gdf_final = gdf_final[~gdf_final.duplicated(subset=['year_ref', 'month_ref', 'geometry', 'area_ha'], keep='first')]
extra_rows = gdf_final[gdf_final.duplicated(subset=['year_ref', 'month_ref', 'geometry'], keep=False)]
extra_rows_no_fire = extra_rows[extra_rows['area_ha'].isna()]
gdf_final = gdf_final.drop(extra_rows_no_fire.index)

# Drop extra columns
gdf_final = gdf_final.drop(['index', 'veg_year', 'value', 'veg_type', 'index_fire', 'year_fire', 'month_fire'], axis=1)

# Rename columns
gdf_final = gdf_final.rename({'year_ref': 'year', 'month_ref': 'month'}, axis=1)

# Add temporal data
gdf_final = gdf_final.merge(df_temporal, how='left', on=['year', 'month'])

# Drop rows without vegetation data
gdf_final = gdf_final[gdf_final['urban_dev'].isna() == False]

# Add columns of interest
gdf_final['fire'] = np.where(gdf_final['area_ha'] > 0, 1, 0)

# One-hot encode month
gdf_final = pd.get_dummies(gdf_final, columns=['month'], prefix='month', drop_first=True)
for col in gdf_final.columns:
    if 'month_' in col:
        gdf_final[col] = gdf_final[col].astype(int)

In [12]:
gdf_final.to_csv("gdf_final.csv", index=False)

## Model Building

In [ ]:
# gdf_final = pd.read_csv('gdf_final.csv')

In [ ]:
data = gdf_final.copy()
del gdf_final
gc.collect()

In [ ]:
# Drop excess zeros to reduce imbalance and dataset size
n_drop = 8 * 10**6
fire_zero_rows = data[data['fire'] == 0]
rows_to_drop = fire_zero_rows.sample(n=min(n_drop, len(fire_zero_rows)), random_state=1)
data_reduced = data.drop(rows_to_drop.index)

# drop the rest of the slope/elev stats that aren't mean
slope_elev_drop = ['slope_min', 'slope_max', 'slope_std', 
                'elev_min', 'elev_max', 'elev_std']
data_reduced = data_reduced.drop(columns=slope_elev_drop)

#### XG Boost

In [ ]:
# sort by year for temporal CV (to mimic future use case)
data_sorted = data_reduced.sort_values('year')

# hold out last 2 years for testing
train_data = data_sorted[data_sorted['year'] <= 2020]
test_data  = data_sorted[data_sorted['year'] > 2020]

X_train = train_data.drop(columns=['fire', 'geometry', 'area_ha', 'year'])
y_train = train_data['fire']

X_test = test_data.drop(columns=['fire', 'geometry', 'area_ha', 'year'])
y_test = test_data['fire']

# compute weights for imbalance
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# base XGBoost model
xgb = XGBClassifier(
    n_estimators=50,
    objective='binary:logistic',
    scale_pos_weight=scale_pos_weight,
    use_label_encoder=False,
    eval_metric='logloss',
    n_jobs=-1,
    random_state=42
)

# temporal CV
tscv = TimeSeriesSplit(n_splits=5)

# small hyperparameter grid for now (kernel keeps crashing)
param_grid = {
    'max_depth': [4, 6],
    'learning_rate': [0.05, 0.1]
}

# grid search with temporal CV
grid = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    cv=tscv,
    scoring='average_precision',  # PR-AUC
    verbose=1
)

# fit model
grid.fit(X_train, y_train)

In [ ]:
# best model
best_model = grid.best_estimator_
print("Best hyperparameters:", grid.best_params_)

# generate predictions
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

# evaluation
print("Classification report:")
print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))
print("PR AUC:", average_precision_score(y_test, y_prob))

In [ ]:
# feature importance
importance = best_model.feature_importances_

feat_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': importance
}).sort_values(by='importance', ascending=False)

print(feat_importance.head(20))  # top 20 features

# Quick plot
plt.figure(figsize=(10,8))
plot_importance(best_model, max_num_features=20, importance_type='weight')
plt.show()